# Handwriting3.12 в Google Colab

Этот блокнот запускает проект в Colab: подключает папку проекта, проверяет GPU, пересобирает датасет, запускает обучение и генерирует траекторию из последнего чекпоинта.

Самый удобный вариант: положить всю папку `Handwriting3.12` в Google Drive по пути `MyDrive/Colab Notebooks/Handwriting3.12`. Тогда чекпоинты будут сохраняться прямо в Drive и не потеряются после перезапуска Colab.

In [ ]:
#@title 1. Проверка GPU
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory: {props.total_memory / 1024**3:.2f} GB")
else:
    print("Включи GPU: Runtime -> Change runtime type -> T4 GPU или другой GPU.")


## Подключение проекта

Оставь `USE_DRIVE = True`, если папка проекта лежит в Google Drive. Если хочешь загрузить ZIP вручную, поставь `USE_DRIVE = False`, запусти ячейку и выбери архив проекта.

In [ ]:
#@title 2. Подключить папку проекта
from pathlib import Path
import os
import sys

USE_DRIVE = True #@param {type:"boolean"}
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/Handwriting3.12" #@param {type:"string"}
ZIP_EXTRACT_DIR = "/content" #@param {type:"string"}

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    if not Path(PROJECT_DIR).exists():
        candidates = [
            "/content/drive/MyDrive/Colab Notebooks/Handwriting3.12",
            "/content/drive/MyDrive/Handwriting3.12",
        ]
        for candidate in candidates:
            if Path(candidate).exists():
                PROJECT_DIR = candidate
                break

    if not Path(PROJECT_DIR).exists():
        raise FileNotFoundError(f"Не нашел папку проекта: {PROJECT_DIR}")
else:
    from google.colab import files
    import zipfile

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("ZIP не загружен.")

    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(ZIP_EXTRACT_DIR)

    guessed_project_dir = Path(ZIP_EXTRACT_DIR) / Path(zip_name).stem
    if guessed_project_dir.exists():
        PROJECT_DIR = str(guessed_project_dir)
    else:
        candidates = [
            p for p in Path(ZIP_EXTRACT_DIR).iterdir()
            if p.is_dir() and (p / "src").exists() and (p / "dataset").exists()
        ]
        if not candidates:
            raise FileNotFoundError("Не нашел папку проекта после распаковки ZIP.")
        PROJECT_DIR = str(candidates[0])

PROJECT_DIR = str(Path(PROJECT_DIR).resolve())
os.chdir(PROJECT_DIR)

src_dir = str(Path(PROJECT_DIR) / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Project dir:", PROJECT_DIR)
print("Current dir:", os.getcwd())


In [ ]:
# 3. Проверка структуры проекта
from pathlib import Path

required_paths = [
    "src/config.py",
    "src/model.py",
    "src/train.py",
    "src/run_training.py",
    "src/generate.py",
    "dataset/converter.py",
    "dataset/jsons",
    "dataset/texts",
]

missing = [path for path in required_paths if not Path(path).exists()]
if missing:
    raise FileNotFoundError("Не хватает файлов или папок: " + ", ".join(missing))

json_count = len(list(Path("dataset/jsons").glob("trajectory_*.json")))
txt_count = len(list(Path("dataset/texts").glob("trajectory_*.txt")))
print(f"JSON files: {json_count}")
print(f"TXT files:  {txt_count}")


In [ ]:
# 4. Библиотеки
# В Colab PyTorch обычно уже установлен. Этой ячейки достаточно для недостающих мелких зависимостей.
!pip install -q tqdm matplotlib numpy


In [ ]:
# 5. Пересобрать dataset/all_trajectories.npz из JSON + TXT
from pathlib import Path

Path("dataset/npzs").mkdir(parents=True, exist_ok=True)
!python dataset/converter.py

npz_path = Path("dataset/all_trajectories.npz")
if not npz_path.exists():
    raise FileNotFoundError("dataset/all_trajectories.npz не создан.")
print("Dataset:", npz_path.resolve())


In [ ]:
#@title 6. Параметры обучения для Colab
from pathlib import Path
import importlib
import config

NUM_EPOCHS = 500 #@param {type:"integer"}
BATCH_SIZE = 2 #@param {type:"integer"}
GRAD_ACCUM_STEPS = 2 #@param {type:"integer"}
LEARNING_RATE = 0.0001 #@param {type:"number"}
SAVE_EVERY = 5 #@param {type:"integer"}
CHECKPOINT_DIR_NAME = "checkpoints_colab" #@param {type:"string"}
AUTO_RESUME = True #@param {type:"boolean"}

# Не перезагружаем config через importlib.reload: ниже мы мягко переопределяем параметры прямо для текущей сессии.
config.num_epochs = int(NUM_EPOCHS)
config.batch_size = int(BATCH_SIZE)
config.grad_accum_steps = int(GRAD_ACCUM_STEPS)
config.learning_rate = float(LEARNING_RATE)
config.save_every = int(SAVE_EVERY)
config.checkpoints = str(Path(PROJECT_DIR) / CHECKPOINT_DIR_NAME)
config.auto_resume = bool(AUTO_RESUME)
config.resume_checkpoint = None
config.progress_mode = "live"
config.progress_mininterval = 1.0
config.progress_ncols = 120
config.num_workers = 2
config.pin_memory = config.device.type == "cuda"
config.empty_cache_each_epoch = True

Path(config.checkpoints).mkdir(parents=True, exist_ok=True)

print("Device:", config.device)
print("Epochs:", config.num_epochs)
print("Batch size:", config.batch_size)
print("Grad accumulation:", config.grad_accum_steps)
print("Effective batch:", config.batch_size * config.grad_accum_steps)
print("Dataset:", config.data_path)
print("Checkpoints:", config.checkpoints)


In [ ]:
# 7. Запуск обучения
# Если AUTO_RESUME=True, обучение продолжится с последнего epoch_*.pth в CHECKPOINT_DIR_NAME.
%run src/run_training.py


## Генерация

Эта ячейка берет последний чекпоинт из `CHECKPOINT_DIR_NAME`, генерирует траекторию и сохраняет ее в `generated_trajectory_colab.json`.

In [ ]:
#@title 8. Сгенерировать траекторию из последнего чекпоинта
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

import config
from generate import generate, plot_trajectory
from model import HandwritingSynthesis

INPUT_TEXT = "слово" #@param {type:"string"}
BIAS = 1.5 #@param {type:"number"}
MIN_GEN_LEN = 200 #@param {type:"integer"}
MAX_GEN_LEN = 3000 #@param {type:"integer"}
OUTPUT_JSON = "generated_trajectory_colab.json" #@param {type:"string"}

def latest_checkpoint(checkpoint_dir):
    checkpoint_paths = []
    for path in Path(checkpoint_dir).glob("epoch_*.pth"):
        match = re.fullmatch(r"epoch_(\d+)\.pth", path.name)
        if match:
            checkpoint_paths.append((int(match.group(1)), path))
    if not checkpoint_paths:
        raise FileNotFoundError(f"В {checkpoint_dir} нет epoch_*.pth")
    return max(checkpoint_paths, key=lambda item: item[0])[1]

missing_chars = sorted({ch for ch in INPUT_TEXT if ch not in config.char_to_idx})
if missing_chars:
    print("Предупреждение: этих символов нет в CHAR_SET и они будут заменены пробелами:", missing_chars)

checkpoint_path = latest_checkpoint(config.checkpoints)
print("Checkpoint:", checkpoint_path)

model = HandwritingSynthesis(
    vocab_size=config.vocab_size,
    embed_dim=config.embed_dim,
    lstm_size=config.lstm_size,
    num_layers=config.num_lstm_layers,
    K=config.K,
    n_mixtures=config.n_mixtures,
    kappa_initial_bias=config.kappa_initial_bias,
).to(config.device)

ckpt = torch.load(checkpoint_path, map_location=config.device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])

dxdy_mean = np.asarray(ckpt.get("dxdy_mean", [[0.0, 0.0]]), dtype=np.float32)
dxdy_std = np.asarray(ckpt.get("dxdy_std", [[1.0, 1.0]]), dtype=np.float32)

trajectory = generate(
    model,
    INPUT_TEXT,
    config.char_to_idx,
    max_len=int(MAX_GEN_LEN),
    device=config.device,
    bias=float(BIAS),
    dxdy_mean=dxdy_mean,
    dxdy_std=dxdy_std,
    min_len=int(MIN_GEN_LEN),
)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(trajectory, f, indent=2, ensure_ascii=False)

print(f"Saved {len(trajectory)} points to {OUTPUT_JSON}")
plt.figure(figsize=(12, 4))
plot_trajectory(trajectory, f'Generated: "{INPUT_TEXT}"')


In [ ]:
# 9. Скачать сгенерированный JSON на компьютер
from google.colab import files

files.download("generated_trajectory_colab.json")


## Быстрые подсказки

- Если добавил новые `json` и `txt`, сначала снова запусти ячейку пересборки датасета.
- Если хочешь продолжить обучение после обрыва Colab, оставь `AUTO_RESUME = True`.
- Если хочешь начать заново, поменяй `CHECKPOINT_DIR_NAME` или поставь `AUTO_RESUME = False`.
- Для T4 обычно нормально начинать с `BATCH_SIZE = 2`, `GRAD_ACCUM_STEPS = 2`. Если памяти хватает, можно пробовать `BATCH_SIZE = 4`.